# 🧰 Project 12.4 — Tool Allocation Optimizer
**DSA for Mechatronics · Week 12 — Dynamic Programming**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, time
from functools import lru_cache
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A robot's tool bay has limited weight capacity. We solve four allocation/optimisation
problems using knapsack-style DP:

1. **0/1 Knapsack** — select tools to maximise utility without exceeding weight limit
   (each tool can be taken once)
2. **Unbounded Knapsack** — unlimited supply of each tool type; maximise utility
3. **Minimum coins** — fewest denomination tokens to make a target amount
   (used for time-slot scheduling)
4. **Count ways (target sum ±)** — count the number of ways to assign + or −
   to reach a target energy balance


## Step 1 — Generate tool and coin datasets

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_TOOLS     = random.randint(7, 11)
CAPACITY    = random.randint(25, 45)
N_COINS     = random.randint(3, 5)
COIN_TARGET = random.randint(20, 35)
N_NUMS      = random.randint(6, 10)
SUM_TARGET  = random.randint(1, 8)

TOOLS = [(f"T{i+1}", random.randint(2, 12), random.randint(5, 30))
         for i in range(N_TOOLS)]   # (name, weight, utility)
COINS = sorted(set([1] + [random.randint(2, 12) for _ in range(N_COINS - 1)]))
NUMS  = [random.randint(1, 10) for _ in range(N_NUMS)]

print(f"Tool bay parameters:")
print(f"  Capacity : {CAPACITY} kg")
print(f"  Tools    : {N_TOOLS}")
print(f"  {'Name':<6}  {'Weight':>8}  {'Utility':>8}")
print(f"  {'─'*6}  {'─'*8}  {'─'*8}")
for name, w, u in TOOLS:
    print(f"  {name:<6}  {w:>8}  {u:>8}")
print()
print(f"  Coins        : {COINS}")
print(f"  Coin target  : {COIN_TARGET}")
print(f"  Nums         : {NUMS}")
print(f"  Sum target   : {SUM_TARGET}")


## Step 2 — 0/1 Knapsack (each tool used at most once)

In [ ]:
def knapsack_01(tools, capacity):
    """
    0/1 Knapsack: select subset of items to maximise value without exceeding capacity.
    Each item can be taken at most once.

    dp[i][w] = max utility using items 0..i-1 with weight limit w.
    Base: dp[0][w] = 0  (no items → 0 utility)
    Recurrence (for item i with weight wi and utility ui):
      Don't take item i: dp[i][w] = dp[i-1][w]
      Take item i (if w >= wi): dp[i][w] = dp[i-1][w-wi] + ui
      dp[i][w] = max of both options.

    Answer: dp[n][capacity]
    O(n * capacity) time and space.

    Backtracking: trace back through table to find which items were taken.
    """
    n = len(tools)
    names = [t[0] for t in tools]
    weights = [t[1] for t in tools]
    utils   = [t[2] for t in tools]
    # Build DP table
    dp = [[0]*(capacity+1) for _ in range(n+1)]
    for i in range(1, n+1):
        wi, ui = weights[i-1], utils[i-1]
        for w in range(capacity+1):
            dp[i][w] = dp[i-1][w]                      # don't take
            if w >= wi:
                dp[i][w] = max(dp[i][w], dp[i-1][w-wi] + ui)  # take
    # Backtrack
    taken = []
    w = capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            taken.append(names[i-1])
            w -= weights[i-1]
    taken.reverse()
    return dp[n][capacity], taken, dp

ks_max, ks_taken, ks_dp = knapsack_01(TOOLS, CAPACITY)
tool_dict = {t[0]: t for t in TOOLS}
taken_weight = sum(tool_dict[t][1] for t in ks_taken)
taken_util   = sum(tool_dict[t][2] for t in ks_taken)

print(f"0/1 Knapsack (capacity={CAPACITY}, n={N_TOOLS} tools):")
print()
print(f"  DP table (last 5 weight columns, all tool rows):")
show_w = min(CAPACITY, 5)
w_cols = list(range(CAPACITY - show_w + 1, CAPACITY + 1))
print(f"  {'Tool':<6}  " + "  ".join(f"w={w:3}" for w in w_cols))
print(f"  {'':<6}  " + "  ".join(f"{ks_dp[0][w]:5}" for w in w_cols))
for i in range(1, N_TOOLS+1):
    print(f"  {TOOLS[i-1][0]:<6}  " + "  ".join(f"{ks_dp[i][w]:5}" for w in w_cols))
print()
print(f"  Maximum utility  : {ks_max}")
print(f"  Tools selected   : {ks_taken}")
print(f"  Total weight     : {taken_weight} / {CAPACITY}")
print(f"  Total utility    : {taken_util}  (== {ks_max}  {'✅' if taken_util == ks_max else '❌'})")


## Step 3 — Unbounded Knapsack (unlimited supply)

In [ ]:
def knapsack_unbounded(tools, capacity):
    """
    Unbounded Knapsack: each item can be used an unlimited number of times.

    Key change from 0/1: when taking item i, we stay in the SAME row
    (dp[w-wi] instead of dp_prev[w-wi]) — allows reusing the same item.

    1D DP is natural here:
      dp[w] = max utility achievable with weight exactly w (or up to w).
    Recurrence:
      for each weight w and each item i:
        if w >= wi: dp[w] = max(dp[w], dp[w - wi] + ui)
    Iterate weights outer, items inner (or items outer, weights inner — both work).

    O(n * capacity) time, O(capacity) space.
    """
    weights = [t[1] for t in tools]
    utils   = [t[2] for t in tools]
    dp = [0] * (capacity + 1)
    choice = [-1] * (capacity + 1)    # which item was added at each weight
    for w in range(1, capacity + 1):
        for i, (wi, ui) in enumerate(zip(weights, utils)):
            if w >= wi and dp[w - wi] + ui > dp[w]:
                dp[w] = dp[w - wi] + ui
                choice[w] = i
    # Reconstruct
    taken_ub = []
    w = capacity
    while w > 0 and choice[w] != -1:
        i = choice[w]
        taken_ub.append(tools[i][0])
        w -= tools[i][1]
    taken_ub_weight = sum(tool_dict[t][1] for t in taken_ub)
    taken_ub_util   = sum(tool_dict[t][2] for t in taken_ub)
    return dp[capacity], taken_ub, taken_ub_weight, taken_ub_util

ub_max, ub_taken, ub_weight, ub_util = knapsack_unbounded(TOOLS, CAPACITY)
from collections import Counter
ub_counts = Counter(ub_taken)

print(f"Unbounded Knapsack (capacity={CAPACITY}):")
print()
print(f"  Maximum utility  : {ub_max}")
print(f"  Tools selected   : {dict(ub_counts)}")
print(f"  Total weight     : {ub_weight} / {CAPACITY}")
print(f"  Total utility    : {ub_util}  (== {ub_max}  {'✅' if ub_util == ub_max else '❌'})")
print()
print(f"  Comparison:")
print(f"    0/1 Knapsack  : utility={ks_max}, weight={taken_weight}")
print(f"    Unbounded     : utility={ub_max}, weight={ub_weight}")
print(f"    Unbounded ≥ 0/1: {'✅ YES' if ub_max >= ks_max else '❌ NO (unexpected)'}")


## Step 4 — Minimum coins + Count ways (target sum ±)

In [ ]:
import math

def min_coins(coins, target):
    """
    Minimum number of coins (unlimited supply) to make up target.

    dp[i] = minimum coins needed to make amount i.
    Base: dp[0] = 0  (0 coins for amount 0)
    Recurrence: dp[i] = 1 + min(dp[i - c] for c in coins if c <= i)
    Answer: dp[target] (inf if impossible)
    O(target * n_coins) time, O(target) space.
    """
    dp = [math.inf] * (target + 1)
    dp[0] = 0
    parent = [-1] * (target + 1)
    for i in range(1, target + 1):
        for c in coins:
            if c <= i and dp[i - c] + 1 < dp[i]:
                dp[i] = dp[i - c] + 1
                parent[i] = c
    # Reconstruct
    used = []
    amt = target
    while amt > 0 and parent[amt] != -1:
        used.append(parent[amt]); amt -= parent[amt]
    return (dp[target] if dp[target] != math.inf else -1), used, dp

def count_target_sum(nums, target):
    """
    Target Sum: assign + or − to each number to reach target.
    Count the number of such assignments.

    Mathematical transform: let P = set of + numbers, N = set of − numbers.
      P - N = target,  P + N = total_sum
      → P = (target + total_sum) / 2
    If (target + total_sum) is odd or |target| > total_sum → 0 ways.
    Otherwise: count subsets of nums that sum to P (standard subset-sum count).

    dp[j] = number of ways to select subset summing to j.
    Iterate backwards (0/1 — each num used once).
    O(n * P) time, O(P) space.
    """
    total = sum(nums)
    if abs(target) > total or (target + total) % 2 == 1:
        return 0, []
    s = (target + total) // 2
    dp = [0] * (s + 1)
    dp[0] = 1
    for num in nums:
        for j in range(s, num - 1, -1):   # iterate BACKWARDS (0/1 knapsack)
            dp[j] += dp[j - num]
    return dp[s], dp

coin_min, coins_used, coin_dp = min_coins(COINS, COIN_TARGET)
ways, ways_dp = count_target_sum(NUMS, SUM_TARGET)

print(f"Minimum coins:")
print(f"  Denominations : {COINS}")
print(f"  Target amount : {COIN_TARGET}")
print(f"  Min coins     : {coin_min}")
print(f"  Coins used    : {sorted(coins_used, reverse=True)}")
print(f"  Sum check     : {sum(coins_used)} == {COIN_TARGET}  "
      f"{'✅' if sum(coins_used)==COIN_TARGET else '❌'}")
print()
print(f"  DP table (amounts 0..{min(COIN_TARGET, 15)}):")
print(f"    amount  : {list(range(min(COIN_TARGET,16)))}")
dp_show = [coin_dp[i] if coin_dp[i] != math.inf else -1
           for i in range(min(COIN_TARGET+1, 16))]
print(f"    min cns : {dp_show}")
print()

print(f"Count target sum (±):")
print(f"  Numbers  : {NUMS}")
print(f"  Target   : {SUM_TARGET}")
print(f"  Total sum: {sum(NUMS)}")
print(f"  Ways     : {ways}")
print()
# Show a few explicit assignments for small cases
if N_NUMS <= 10 and ways <= 20:
    found = []
    for mask in range(1 << N_NUMS):
        s = sum(NUMS[i] if (mask >> i) & 1 else -NUMS[i] for i in range(N_NUMS))
        if s == SUM_TARGET:
            signs = ["+" if (mask >> i) & 1 else "-" for i in range(N_NUMS)]
            found.append("  ".join(f"{s}{v}" for s, v in zip(signs, NUMS)))
    print(f"  Explicit assignments ({len(found)}):")
    for f in found[:8]:
        print(f"    {f}")
    if len(found) > 8:
        print(f"    ... and {len(found)-8} more")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " TOOL ALLOCATION OPTIMIZER — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  0/1 KNAPSACK" + " "*(W-14) + "║")
print(f"║  {'Tools (n)':<28}: {N_TOOLS:<26}║")
print(f"║  {'Capacity':<28}: {CAPACITY} kg{'':<22}║")
print(f"║  {'Max utility':<28}: {ks_max:<26}║")
print(f"║  {'Tools taken':<28}: {str(ks_taken):<26}║")
print(f"║  {'Weight used':<28}: {taken_weight} / {CAPACITY}{'':<18}║")
print("╠" + "═"*W + "╣")
print("║  UNBOUNDED KNAPSACK" + " "*(W-20) + "║")
print(f"║  {'Max utility':<28}: {ub_max:<26}║")
print(f"║  {'Tools (with counts)':<28}: {str(dict(ub_counts)):<26}║")
print(f"║  {'Unbounded ≥ 0/1':<28}: {'YES ✅' if ub_max >= ks_max else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  MINIMUM COINS" + " "*(W-15) + "║")
print(f"║  {'Denominations':<28}: {str(COINS):<26}║")
print(f"║  {'Target':<28}: {COIN_TARGET:<26}║")
print(f"║  {'Min coins needed':<28}: {coin_min:<26}║")
print("╠" + "═"*W + "╣")
print("║  TARGET SUM (±)" + " "*(W-16) + "║")
print(f"║  {'Numbers':<28}: {str(NUMS):<26}║")
print(f"║  {'Target':<28}: {SUM_TARGET:<26}║")
print(f"║  {'Number of ways':<28}: {ways:<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about DP in this context?

*Your answer here:*

---

**Q2 — Memoization vs Tabulation — which did you find clearer, and why?**
Describe the trade-offs between top-down (memoization) and bottom-up (tabulation) for one of the problems in this project.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
